# 淘宝用户行为漏斗分析

本 Notebook 使用 `chunksize` 分块读取 `UserBehavior.csv` 全量数据，统计行为层面和用户层面的漏斗指标，并输出结果表到 `data/processed/`。

原始数据没有表头，字段为：`user_id`、`item_id`、`category_id`、`behavior_type`、`timestamp`。

## 1. 导入工具包与基础配置

In [ ]:
# 导入数据分析常用工具包
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

# 设置 matplotlib 中文字体，避免图表中的中文乱码
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

# E 盘项目根目录：所有输入、输出都放在这个项目中
PROJECT_ROOT = Path(r"E:\ecommerce-user-growth-analysis")

# 原始数据路径：使用 E 盘项目中的全量 CSV 文件
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "UserBehavior.csv"

# 处理后数据保存目录：结果表统一保存到 E 盘项目的 data/processed/
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# CSV 没有表头，因此手动指定列名
COLUMN_NAMES = ["user_id", "item_id", "category_id", "behavior_type", "timestamp"]

# 分块大小：每次只读取一部分数据，避免一次性读入内存
CHUNK_SIZE = 1_000_000

# 四类行为的展示顺序
BEHAVIOR_ORDER = ["pv", "fav", "cart", "buy"]

# 四类行为的中文名称
BEHAVIOR_LABELS = {
    "pv": "浏览",
    "fav": "收藏",
    "cart": "加购",
    "buy": "购买",
}

print(f"原始数据路径：{RAW_DATA_PATH}")
print(f"结果保存目录：{PROCESSED_DIR}")

## 2. 分块读取全量数据并累计统计

这里不会一次性把全量 CSV 读入内存，而是每次读取 `CHUNK_SIZE` 行。行为数量用计数器累加，用户数量用集合记录去重后的用户 ID。

In [ ]:
# 如果原始文件不存在，提前给出清晰提示
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"找不到原始数据文件：{RAW_DATA_PATH}")

# 初始化行为数量统计，每种行为先从 0 开始
behavior_counts = pd.Series(0, index=BEHAVIOR_ORDER, dtype="int64")

# 初始化用户集合，用集合可以自动去重
behavior_user_sets = {behavior: set() for behavior in BEHAVIOR_ORDER}

# 记录总行数，方便检查是否完整读取
total_rows = 0

# 使用 chunksize 分块读取全量数据
reader = pd.read_csv(
    RAW_DATA_PATH,
    names=COLUMN_NAMES,
    header=None,
    chunksize=CHUNK_SIZE,
)

# 逐块处理数据
for chunk_no, chunk in enumerate(reader, start=1):
    # 累计当前分块的行数
    total_rows += len(chunk)

    # 统计当前分块中四类行为的数量，并累加到全量结果
    chunk_behavior_counts = chunk["behavior_type"].value_counts()
    behavior_counts = behavior_counts.add(chunk_behavior_counts, fill_value=0).astype("int64")

    # 按行为类型提取用户 ID，并更新到对应集合中完成去重
    for behavior in BEHAVIOR_ORDER:
        user_ids = chunk.loc[chunk["behavior_type"] == behavior, "user_id"].unique()
        behavior_user_sets[behavior].update(user_ids)

    # 每处理一个分块打印一次进度，方便观察运行状态
    print(f"已处理第 {chunk_no} 个分块，累计读取 {total_rows:,} 行")

# 保证行为数量按照指定顺序展示
behavior_counts = behavior_counts.reindex(BEHAVIOR_ORDER).fillna(0).astype("int64")

# 统计每类行为对应的去重用户数
user_counts = pd.Series(
    {behavior: len(behavior_user_sets[behavior]) for behavior in BEHAVIOR_ORDER},
    dtype="int64",
).reindex(BEHAVIOR_ORDER)

print("\n全量数据读取完成")
print(f"总行数：{total_rows:,}")

## 3. 行为层面漏斗指标

行为层面关注的是不同类型行为发生的次数，例如浏览次数、收藏次数、加购次数和购买次数。

In [ ]:
# 定义安全除法函数，避免分母为 0 时程序报错
def safe_divide(numerator, denominator):
    """计算转化率；如果分母为 0，则返回 0。"""
    return numerator / denominator if denominator else 0


# 取出各行为数量，变量名更直观，便于初学者理解
pv_count = int(behavior_counts["pv"])
fav_count = int(behavior_counts["fav"])
cart_count = int(behavior_counts["cart"])
buy_count = int(behavior_counts["buy"])
total_behavior_count = int(behavior_counts.sum())

# 计算行为数量占比
behavior_ratios = behavior_counts / total_behavior_count

# 计算题目要求的行为层面转化率
pv_to_fav_rate = safe_divide(fav_count, pv_count)
pv_to_cart_rate = safe_divide(cart_count, pv_count)
pv_to_buy_rate = safe_divide(buy_count, pv_count)
cart_to_buy_rate = safe_divide(buy_count, cart_count)
fav_to_buy_rate = safe_divide(buy_count, fav_count)

# 汇总成行为漏斗结果表
behavior_funnel_metrics = pd.DataFrame(
    {
        "behavior_type": BEHAVIOR_ORDER,
        "behavior_name": [BEHAVIOR_LABELS[x] for x in BEHAVIOR_ORDER],
        "behavior_count": [pv_count, fav_count, cart_count, buy_count],
        "behavior_ratio": [behavior_ratios[x] for x in BEHAVIOR_ORDER],
        "conversion_from_pv": [1, pv_to_fav_rate, pv_to_cart_rate, pv_to_buy_rate],
        "conversion_to_buy": [pv_to_buy_rate, fav_to_buy_rate, cart_to_buy_rate, 1],
    }
)

# 补充字段说明，方便以后直接读 CSV 时理解含义
behavior_funnel_metrics["note"] = [
    "浏览行为数量；conversion_to_buy 表示 pv 到 buy",
    "收藏行为数量；conversion_to_buy 表示 fav 到 buy",
    "加购行为数量；conversion_to_buy 表示 cart 到 buy",
    "购买行为数量；conversion_from_pv 表示 pv 到 buy",
]

# 保存行为漏斗结果表
behavior_output_path = PROCESSED_DIR / "behavior_funnel_metrics.csv"
behavior_funnel_metrics.to_csv(behavior_output_path, index=False, encoding="utf-8-sig")

# 展示结果表
display(behavior_funnel_metrics)
print(f"行为漏斗指标已保存：{behavior_output_path}")

## 4. 用户层面漏斗指标

用户层面关注的是发生过某类行为的去重用户数，例如有浏览行为的用户数、有购买行为的用户数。

In [ ]:
# 取出各行为对应的去重用户数
pv_user_count = int(user_counts["pv"])
fav_user_count = int(user_counts["fav"])
cart_user_count = int(user_counts["cart"])
buy_user_count = int(user_counts["buy"])

# 计算题目要求的用户层面转化率
pv_user_to_buy_user_rate = safe_divide(buy_user_count, pv_user_count)
cart_user_to_buy_user_rate = safe_divide(buy_user_count, cart_user_count)
fav_user_to_buy_user_rate = safe_divide(buy_user_count, fav_user_count)

# 汇总成用户漏斗结果表
user_funnel_metrics = pd.DataFrame(
    {
        "behavior_type": BEHAVIOR_ORDER,
        "behavior_name": [BEHAVIOR_LABELS[x] for x in BEHAVIOR_ORDER],
        "user_count": [pv_user_count, fav_user_count, cart_user_count, buy_user_count],
        "conversion_to_buy_user": [
            pv_user_to_buy_user_rate,
            fav_user_to_buy_user_rate,
            cart_user_to_buy_user_rate,
            1,
        ],
    }
)

# 补充字段说明，方便以后直接读 CSV 时理解含义
user_funnel_metrics["note"] = [
    "有浏览行为的用户数；conversion_to_buy_user 表示浏览用户到购买用户",
    "有收藏行为的用户数；conversion_to_buy_user 表示收藏用户到购买用户",
    "有加购行为的用户数；conversion_to_buy_user 表示加购用户到购买用户",
    "有购买行为的用户数",
]

# 保存用户漏斗结果表
user_output_path = PROCESSED_DIR / "user_funnel_metrics.csv"
user_funnel_metrics.to_csv(user_output_path, index=False, encoding="utf-8-sig")

# 展示结果表
display(user_funnel_metrics)
print(f"用户漏斗指标已保存：{user_output_path}")

## 5. 绘制漏斗图

In [ ]:
# 定义通用漏斗图函数，行为漏斗和用户漏斗都可以复用
def plot_funnel(values, labels, title, color):
    """绘制横向漏斗图。"""
    fig, ax = plt.subplots(figsize=(10, 5))

    # 横向条形图从上到下展示漏斗阶段
    bars = ax.barh(labels, values, color=color)
    ax.invert_yaxis()

    # 在条形右侧标出具体数量和相对首环节的比例
    first_value = values[0]
    max_value = max(values) if len(values) > 0 else 0
    for bar, value in zip(bars, values):
        rate = safe_divide(value, first_value)
        ax.text(
            value + max_value * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{value:,}（较首环节 {rate:.2%}）",
            va="center",
            fontsize=10,
        )

    # 设置标题和坐标轴样式
    ax.set_title(title, fontsize=15, pad=12)
    ax.set_xlabel("数量")
    ax.set_xlim(0, max_value * 1.25 if max_value else 1)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.show()


# 行为数量漏斗图
plot_funnel(
    behavior_funnel_metrics["behavior_count"].tolist(),
    behavior_funnel_metrics["behavior_name"].tolist(),
    "行为数量漏斗图",
    "#4C78A8",
)

# 用户数量漏斗图
plot_funnel(
    user_funnel_metrics["user_count"].tolist(),
    user_funnel_metrics["behavior_name"].tolist(),
    "用户数量漏斗图",
    "#F58518",
)

## 6. 业务解读

In [ ]:
# 根据计算结果自动生成业务解读，保证结论和实际数据一致
behavior_drop_rates = {
    "浏览到收藏": 1 - pv_to_fav_rate,
    "浏览到加购": 1 - pv_to_cart_rate,
    "浏览到购买": 1 - pv_to_buy_rate,
    "加购到购买": 1 - cart_to_buy_rate,
    "收藏到购买": 1 - fav_to_buy_rate,
}

# 找出流失率最高的环节
largest_drop_stage = max(behavior_drop_rates, key=behavior_drop_rates.get)
largest_drop_rate = behavior_drop_rates[largest_drop_stage]

# 判断用户更偏向收藏还是加购
if fav_count > cart_count:
    preference_text = f"从行为次数看，收藏次数（{fav_count:,}）高于加购次数（{cart_count:,}），用户更偏向先收藏。"
elif cart_count > fav_count:
    preference_text = f"从行为次数看，加购次数（{cart_count:,}）高于收藏次数（{fav_count:,}），用户更偏向直接加购。"
else:
    preference_text = f"从行为次数看，收藏次数和加购次数相同，均为 {fav_count:,}。"

# 判断增长优化优先级
if pv_to_buy_rate < min(pv_to_fav_rate, pv_to_cart_rate):
    growth_text = "从增长角度看，应优先优化从浏览到购买的承接效率，例如强化商品详情页信息、优惠触达、信任背书和下单路径。"
elif cart_to_buy_rate < fav_to_buy_rate:
    growth_text = "从增长角度看，应优先优化加购后的购买转化，例如购物车召回、价格提醒、优惠券和结算流程。"
else:
    growth_text = "从增长角度看，应优先优化收藏后的购买转化，例如收藏夹召回、降价提醒和个性化推荐。"

# 使用 Markdown 展示最终业务解读
display(
    Markdown(
        f"""
### 业务解读

1. **哪个环节流失最明显？**  
   行为层面流失最明显的是 **{largest_drop_stage}**，流失率约为 **{largest_drop_rate:.2%}**。这说明用户从前序行为进入该后序行为时损耗较大，需要重点关注该环节的用户动机和阻碍因素。

2. **用户更偏向收藏还是加购？**  
   {preference_text} 如果收藏明显高于加购，说明用户仍处于比较和犹豫阶段；如果加购明显高于收藏，说明用户购买意向更强，但还需要推动其完成支付。

3. **从增长角度应该优先优化哪个环节？**  
   {growth_text} 同时建议结合用户层面的转化率观察：浏览用户到购买用户代表整体转化效率，加购用户到购买用户和收藏用户到购买用户代表高意向用户的转化效率。
        """
    )
)